# Phase 1 — FP32 Baseline (Kaggle GPU)

**Goal:** establish ground-truth FP32 numbers for SmolLM2-1.7B that every later quantization experiment is measured against.

**This notebook produces:**
1. `results/baseline/baseline_results.json` — FP32 perplexity on WikiText-103 test, plus optional lm-eval scores.
2. `results/baseline/fp32_logits.pt` — saved FP32 output logits used as the reference distribution for KL-divergence in notebooks 02–05.

**Important:** when this notebook finishes, click **Save Version → Save & Run All**. After it commits, go to the version's *Output* tab and click **Add as Dataset** so notebooks 02–05 can mount the FP32 logits without re-running this notebook.

**Estimated time on a T4 GPU:** ~25 min (perplexity + logit save). Add ~3 h if `RUN_BENCHMARKS=True`.

## 1. Setup — clone repo, install package, GPU check

In [ ]:
# === Kaggle setup ===
# Two options to bring in the source code:
#   A) git clone (default; needs Internet enabled in Kaggle settings)
#   B) mount the repo as a Kaggle Dataset and copy it (offline-friendly)
#
# Edit GITHUB_URL to your fork if needed.
import os, sys, shutil, subprocess

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"   # only used if option B

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        print(f"Copying repo from Kaggle dataset: {REPO_DATASET}")
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        print(f"Cloning repo from: {GITHUB_URL}")
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo:", REPO_DIR)
print("CWD :", os.getcwd())

In [ ]:
# Editable install — registers `src` as an importable package and installs deps.
# `-q` keeps Kaggle output clean; remove if you need to debug install issues.
!pip install -q -e ".[eval,viz]"

In [ ]:
import torch

torch.manual_seed(42)

print(f"torch    : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")
    print(f"Compute  : {props.major}.{props.minor}")
else:
    print("WARNING — running on CPU. FP32 SmolLM2-1.7B will be extremely slow.")

## 2. Configuration

All Kaggle-specific knobs in one cell. Edit and re-run only this cell to change behaviour without re-running the setup.

In [ ]:
from pathlib import Path

MODEL_NAME      = "HuggingFaceTB/SmolLM2-1.7B"
CACHE_DIR       = f"{WORKING_DIR}/models/base"
OUTPUT_DIR      = f"{WORKING_DIR}/results/baseline"

# Perplexity is reported at the standard SmolLM2 max context (2048).
EVAL_SEQ_LEN    = 2048
EVAL_BATCH      = 2          # T4 16GB — keep low for FP32 forward pass at 2048 tokens

# Logits saved for KL divergence in later notebooks. Use 512-tok seqs to match the
# reduced training seq_length we'll use in QAT notebooks.
LOGIT_SEQ_LEN   = 512
LOGIT_SAMPLES   = 32         # 32 × 512 × 49152 × 2B fp16 ≈ 1.6 GB

# lm-evaluation-harness is slow (~3 h on T4 for full suite). Off by default.
RUN_BENCHMARKS  = False
BENCHMARK_TASKS = ["hellaswag", "arc_challenge", "piqa"]   # cheap subset

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print(f"Model      : {MODEL_NAME}")
print(f"Cache dir  : {CACHE_DIR}")
print(f"Output dir : {OUTPUT_DIR}")

## 3. Download SmolLM2-1.7B

First call downloads ~6.5 GB of FP32 weights into `CACHE_DIR`. Subsequent runs read from cache.

In [ ]:
from src.training.baseline import download_model

download_model(model_name=MODEL_NAME, cache_dir=CACHE_DIR)

## 4. FP32 perplexity + saved logits

`run_baseline()` does the full Phase 1 pass:
1. Load FP32 model on GPU.
2. Compute perplexity on the WikiText-103 **test** split (`seq_length=2048`).
3. Save logits from the WikiText-103 **train** split (`seq_length=512`, 32 samples) for KL divergence.

Step 3 produces the artifact every later notebook needs.

In [ ]:
from src.training.baseline import run_baseline

results = run_baseline(
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
    output_dir=OUTPUT_DIR,
    device_str="cuda",
    seq_length=EVAL_SEQ_LEN,
    eval_batch_size=EVAL_BATCH,
    save_logits=True,
    num_logit_samples=LOGIT_SAMPLES,
    run_benchmarks=False,           # benchmark cell below if you want them
)

print("\nBaseline summary:")
for k, v in results.items():
    print(f"  {k:20s}  {v}")

**Note on logit sequence length:** `run_baseline` saves logits at `seq_length=EVAL_SEQ_LEN`. For Kaggle we want them at `LOGIT_SEQ_LEN=512` to match QAT training. The cell below regenerates the logits file at the smaller length — comment it out if you want full-2048 logits.

In [ ]:
# Regenerate logits at LOGIT_SEQ_LEN (512) for KL divergence with reduced-context QAT runs.
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM
from src.utils.data_loader import _load_and_chunk, get_tokenizer
from src.utils.metrics import save_fp32_logits

device = torch.device("cuda")
tokenizer = get_tokenizer(MODEL_NAME, CACHE_DIR)
ds = _load_and_chunk("wikitext-103-raw-v1", "train", tokenizer,
                     seq_length=LOGIT_SEQ_LEN, num_samples=LOGIT_SAMPLES)
loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, cache_dir=CACHE_DIR, torch_dtype=torch.float32
).to(device).eval()

logits_path = Path(OUTPUT_DIR) / "fp32_logits.pt"
save_fp32_logits(model, loader, logits_path, device, num_samples=LOGIT_SAMPLES)

size_gb = logits_path.stat().st_size / 1e9
print(f"Saved {LOGIT_SAMPLES} logit sequences @ seq_len={LOGIT_SEQ_LEN} "
      f"-> {logits_path} ({size_gb:.2f} GB)")

del model
torch.cuda.empty_cache()

## 5. (Optional) lm-evaluation-harness benchmarks

Set `RUN_BENCHMARKS=True` in the configuration cell to run this. Full suite (MMLU + HellaSwag + ARC + PIQA + GSM8K) takes ~3 h on a T4. The default `BENCHMARK_TASKS` list omits MMLU and GSM8K to keep it under an hour.

Skip this section if you're just establishing FP32 perplexity for the QAT comparisons — the FP32 benchmark numbers can be filled in later from the published SmolLM2 paper or a separate evaluation run.

In [ ]:
if RUN_BENCHMARKS:
    from src.utils.metrics import run_lm_eval
    bench = run_lm_eval(
        model_path=MODEL_NAME,
        output_path=Path(OUTPUT_DIR) / "lm_eval",
        tasks=BENCHMARK_TASKS,
        batch_size=8,
        device="cuda",
    )
    results["lm_eval"] = bench
    for task, r in bench.items():
        acc, err = r.get("acc"), r.get("acc_stderr")
        acc_s = f"{acc:.4f}" if acc is not None else "   n/a"
        err_s = f"{err:.4f}" if err is not None else "   n/a"
        print(f"  {task:20s}  acc={acc_s}  ±{err_s}")
else:
    print("Skipped (RUN_BENCHMARKS=False).")

## 6. Persist final results

In [ ]:
import json

summary_path = Path(OUTPUT_DIR) / "baseline_results.json"
with summary_path.open("w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Wrote {summary_path}")
print(json.dumps(results, indent=2, default=str))

In [ ]:
# Sanity check: confirm the artifacts the next notebooks expect.
!ls -lh {OUTPUT_DIR}

## Next steps

1. **Save Version** in Kaggle (top-right) → *Save & Run All*.
2. After commit, open the version's **Output** tab and click **Add as Dataset**. Name it e.g. `sqat-baseline`.
3. In notebooks 02–05, add this dataset as input — they mount it at `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt`.

Proceed to `02_ptq.ipynb`.